In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
!pip install catboost

%matplotlib inline
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns="Order_ID", axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
missing_columns = ["Weather", "Traffic_Level", "Time_of_Day", "Courier_Experience_yrs", "Delivery_Time"]
df_clean = df.dropna(subset=missing_columns).copy()
check_missing_values(df_clean)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
numerical_cols = df_clean.select_dtypes(exclude=["object"]).columns

if len(categorical_cols) > 0:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    encoded = ohe.fit_transform(df_clean[categorical_cols])

    encoded_df = pd.DataFrame(
        encoded,
        columns=ohe.get_feature_names_out(categorical_cols),
        index=df_clean.index
    )

    df_clean = pd.concat([df_clean[numerical_cols], encoded_df], axis=1)

df_clean.head()



In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# We don't need to balance for this question.

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean["Delivery_Time"].astype(float)

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_scores = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    print(f"MAE: {mae:.4f}")

print("\n=== Final Result ===")
print(f"Average MAE across folds: {np.mean(mae_scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import numpy as np


importances = model.feature_importances_
feature_names = X.columns

indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(len(importances)), importances[indices], align="center")
plt.xticks(range(len(importances)), feature_names[indices], rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(8, 5))
plt.hist(y_pred, bins=30, color='blue', edgecolor='black')
plt.title("Predicted Delivery Time Histogram")
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import numpy as np

X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean["Delivery_Time"].astype(float)

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_scores = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)

    cb = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="MAE",
        verbose=0,
        random_seed=42
    )
    cb.fit(X_train, y_train)


    preds_rf = rf.predict(X_test)
    preds_cb = cb.predict(X_test)

    preds_avg = (preds_rf + preds_cb) / 2

    mae = mean_absolute_error(y_test, preds_avg)
    mae_scores.append(mae)

    print(f"Ensemble MAE: {mae:.4f}")

print("\n=== Ensemble Final Result ===")
print(f"Average Ensemble MAE across folds: {np.mean(mae_scores):.4f}")

